# Dataset Audit

Prüft `data/processed/metadata.parquet` und die Split-Listen auf Konsistenz:
keine Sequenz in zwei Splits, der Split aus Seed und Metadaten reproduzierbar,
keine fehlenden Werte in den Spalten, die die Pipeline braucht. Dazu die
Kennzahlen, die man zum Einordnen der Ergebnisse braucht — Sequenzlängen,
Aufnahmejahre, Fotografen, Nähe zwischen Query und Datenbank.

Schreibt `results/dataset_audit.json`; ein verletztes Kriterium bricht ab.

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, paths
from src.pairs import query_database_pairs
from src.split import draw_split, read_split_lists

plt.rcParams["figure.dpi"] = 300

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)
PROCESSED_DIR = PATHS.processed
FIGURE_DIR = PATHS.figures / "dataset"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_PATH = PATHS.dataset_audit

metadata = pd.read_parquet(PROCESSED_DIR / "metadata.parquet")
train_sequences, database_sequences, query_sequences = read_split_lists(PROCESSED_DIR)

print(f"Metadaten: {len(metadata):,} Bilder, {metadata['sequence_id'].nunique():,} Sequenzen")
print(f"Split-Listen: train {len(train_sequences):,} / database {len(database_sequences):,} "
      f"/ query {len(query_sequences):,} Sequenzen")

## Grundstatistik und fehlende Werte

Median deutlich unter dem Mittelwert heißt: wenige sehr lange Fahrten ziehen
den Mittelwert hoch. Sequenzen mit einem Bild bleiben im Datensatz, sie
werden nur gezählt. Fehlende Werte sind kein Löschgrund, aber die Spalten,
die die Pipeline braucht, müssen vollständig sein.

In [ ]:
images_per_sequence = metadata["sequence_id"].value_counts()

print(f"Bilder je Sequenz:  Median {images_per_sequence.median():.0f}, "
      f"Mittel {images_per_sequence.mean():.0f}, "
      f"min {images_per_sequence.min()}, max {images_per_sequence.max():,}")
print(f"Sequenzen mit einem Bild: {(images_per_sequence == 1).sum()}")
print()
print(f"{'Spalte':<15}{'fehlend':>9}{'Anteil':>9}")
for spalte in metadata.columns:
    n = int(metadata[spalte].isna().sum())
    print(f"{spalte:<15}{n:>9,}{n / len(metadata):>9.2%}")

# Mapillary kodiert eine unbekannte Blickrichtung als -1.
print(f"\ncompass_angle unbekannt (-1): {(metadata['compass_angle'] < 0).sum()}, "
      f"ueber 360: {(metadata['compass_angle'] > 360).sum()}")
print(f"Panoramen: {metadata['is_pano'].sum()}")

## Sequenzgrößen

Die zehn längsten und kürzesten Sequenzen, dann das Histogramm — auf 1.000
Bilder je Sequenz begrenzt, sonst ist die Verteilung nicht lesbar.

In [ ]:
print("10 laengste Sequenzen:")
print(images_per_sequence.head(10).rename_axis(None).to_string())
print("\n10 kuerzeste Sequenzen:")
print(images_per_sequence.sort_values().head(10).rename_axis(None).to_string())

plt.figure(figsize=(10, 6))
plt.xlim(0, 1000)
plt.hist(images_per_sequence, bins=50)
plt.xlabel("Bilder pro Sequenz")
plt.ylabel("Anzahl Sequenzen")
plt.title("Verteilung der Sequenzgroessen")
plt.savefig(FIGURE_DIR / "sequence_sizes.png")
plt.show()

## Split-Konsistenz

Drei Prüfungen, jede bricht ab, wenn sie scheitert: keine Sequenz in zwei
Listen; jede Sequenz der Metadaten in genau einem Split, und zwar dem aus
der Liste; und der Split ist aus Seed und Metadaten reproduzierbar — das
heißt, die Listen wurden nicht von Hand verändert.

In [ ]:
train_set, database_set, query_set = map(set, (train_sequences, database_sequences, query_sequences))
ueberschneidung = (train_set & database_set) | (train_set & query_set) | (database_set & query_set)
assert not ueberschneidung, f"Sequenz-Leakage zwischen Splits: {sorted(ueberschneidung)[:5]}"

erwartet = ({s: "train" for s in train_set} | {s: "database" for s in database_set}
            | {s: "query" for s in query_set})
je_sequenz = metadata.groupby("sequence_id")["split"].agg(["nunique", "first"])
assert (je_sequenz["nunique"] == 1).all(), "Eine Sequenz liegt in mehreren Splits"
falsch = je_sequenz[je_sequenz["first"] != je_sequenz.index.map(erwartet)]
assert falsch.empty, f"{len(falsch)} Sequenzen widersprechen den Split-Listen"
fehlt_in_listen = set(je_sequenz.index) - set(erwartet)
assert not fehlt_in_listen, f"{len(fehlt_in_listen)} Sequenzen ohne Eintrag in den Split-Listen"

vpr = CFG["vpr"]
neu = draw_split(je_sequenz.index, int(vpr["split_seed"]),
                 float(vpr["train_fraction"]), float(vpr["database_fraction"]))
reproduzierbar = [list(a) == list(b) for a, b in
                  zip(neu, (train_sequences, database_sequences, query_sequences))]

images_per_split = metadata["split"].value_counts()
print("Bilder je Split:")
print(images_per_split.rename_axis(None).to_string())
print(f"\nSplit aus Seed {vpr['split_seed']} reproduzierbar: "
      f"train {reproduzierbar[0]}, database {reproduzierbar[1]}, query {reproduzierbar[2]}")
nicht_in_metadata = set(erwartet) - set(je_sequenz.index)
if nicht_in_metadata:
    print(f"{len(nicht_in_metadata)} Sequenzen aus den Listen fehlen in den Metadaten "
          "(Mapillary hat sie seit dem Split entfernt)")

## Zeitliche Verteilung

`captured_at` ist ein Millisekunden-Zeitstempel. Große Zeitabstände zwischen
Query und Datenbank sind kein Fehler — sie sind das, woran ein VPR-System
gemessen wird.

In [ ]:
metadata["jahr"] = pd.to_datetime(metadata["captured_at"], unit="ms", errors="coerce").dt.year
jahre = metadata["jahr"].value_counts().sort_index()
print("Bilder je Aufnahmejahr:")
print(jahre.rename_axis(None).to_string())
print()
print(metadata.groupby(["jahr", "split"]).size().unstack(fill_value=0).to_string())

plt.figure(figsize=(12, 6))
plt.bar(jahre.index, jahre.values)
plt.xlabel("Jahr")
plt.ylabel("Anzahl Bilder")
plt.title("Mapillary-Bilder je Aufnahmejahr")
plt.xticks(jahre.index, rotation=45)
plt.savefig(FIGURE_DIR / "images_per_year.png")
plt.show()

## Räumliche Verteilung

Alle Bildpositionen und, daneben, Datenbank gegen Query — dort sieht man,
wo die Referenz dünn ist.

In [ ]:
images_gdf = gpd.GeoDataFrame(
    metadata, geometry=gpd.points_from_xy(metadata["lon"], metadata["lat"]), crs="EPSG:4326"
)

ax = images_gdf.plot(figsize=(10, 10), markersize=1)
ax.set_title("Mapillary-Abdeckung Osnabrueck")
plt.savefig(FIGURE_DIR / "coverage_map.png")
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
images_gdf[images_gdf["split"] == "database"].plot(ax=ax, markersize=1, label="Database")
images_gdf[images_gdf["split"] == "query"].plot(ax=ax, markersize=1, label="Query")
ax.set_title("Database gegen Query")
ax.legend()
plt.savefig(FIGURE_DIR / "database_vs_query.png")
plt.show()

## Nähe zwischen Query und Datenbank, Fotografen

Wie viele Anfragen haben überhaupt ein Datenbankbild im Umkreis der Ground
Truth — und wie viele davon eines vom selben Fotografen am selben Tag? Das
zweite ist die milde Form von Leakage, die ein Sequenz-Split nicht
ausschließt: dieselbe Person fährt dieselbe Straße zweimal. Dazu, wie
stark der Datensatz an einzelnen Fotografen hängt.

In [ ]:
RADIUS_M = float(CFG["vpr"]["uncertain_radius_m"])
POSITIVE_M = float(CFG["vpr"]["positive_radius_m"])

meta_idx = metadata.set_index("image_id")[["captured_at", "creator_id", "sequence_id"]]
paare = query_database_pairs(metadata, RADIUS_M)
paare = paare.join(meta_idx.add_prefix("q_"), on="query_image_id")
paare = paare.join(meta_idx.add_prefix("db_"), on="database_image_id")
paare["gleicher_fotograf"] = paare["q_creator_id"] == paare["db_creator_id"]
paare["tage"] = (paare["q_captured_at"] - paare["db_captured_at"]).abs() / 86_400_000
paare["gleicher_tag"] = paare["gleicher_fotograf"] & (paare["tage"] < 1)

n_query = int((metadata["split"] == "query").sum())
je_query = paare.groupby("query_image_id").agg(
    n=("database_image_id", "size"), gleicher_tag=("gleicher_tag", "any"),
    anderer_fotograf=("gleicher_fotograf", lambda s: (~s).any()), tage_min=("tage", "min"),
)
n_loesbar = len(je_query)
print(f"Anfragen mit Datenbankbild im Umkreis von {RADIUS_M:g} m: {n_loesbar:,} von {n_query:,} "
      f"({n_loesbar / n_query:.1%})")
print(f"  davon mit Nachbar vom selben Fotografen am selben Tag: {je_query['gleicher_tag'].mean():.1%}")
print(f"  davon mit Nachbar von einem anderen Fotografen:        {je_query['anderer_fotograf'].mean():.1%}")
print(f"  Nachbarn je loesbarer Anfrage: Median {je_query['n'].median():.0f}")
print(f"  Zeitabstand zum naechsten Referenzbild: Median {je_query['tage_min'].median():.0f} Tage")
print(f"Paare unter {POSITIVE_M:g} m: {(paare['distance_m'] <= POSITIVE_M).sum():,}")

fotografen = metadata["creator_id"].value_counts(normalize=True)
print(f"\nFotografen: {len(fotografen)}; groesster Anteil {fotografen.iloc[0]:.1%}, "
      f"die drei groessten zusammen {fotografen.iloc[:3].sum():.1%}")

## Audit-Fazit

Der Datensatz gilt als gültig, wenn keine Sequenz in zwei Splits liegt, die
Splits der Metadaten den Listen entsprechen, der Split reproduzierbar ist,
und die Pflichtspalten keine fehlenden Werte haben. Alles andere sind
Kennzahlen. `run.py` erkennt an der JSON, dass 02 gelaufen ist.

In [ ]:
pflicht = ["image_id", "sequence_id", "captured_at", "lat", "lon", "split"]
fehlend_pflicht = int(metadata[pflicht].isna().sum().sum())
assert fehlend_pflicht == 0, f"{fehlend_pflicht} fehlende Werte in Pflichtspalten"

audit = {
    "datum": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "bilder": int(len(metadata)),
    "sequenzen": int(metadata["sequence_id"].nunique()),
    "sequenzen_mit_einem_bild": int((images_per_sequence == 1).sum()),
    "bilder_je_sequenz_median": float(images_per_sequence.median()),
    "bilder_je_split": {k: int(v) for k, v in images_per_split.items()},
    "split_reproduzierbar": bool(all(reproduzierbar)),
    "sequenz_leakage": 0,
    "fehlende_pflichtwerte": fehlend_pflicht,
    "panoramen": int(metadata["is_pano"].sum()),
    "compass_unbekannt": int((metadata["compass_angle"] < 0).sum()),
    "aufnahmejahre": {int(k): int(v) for k, v in jahre.items()},
    "fotografen": int(len(fotografen)),
    "anteil_groesster_fotograf": float(fotografen.iloc[0]),
    "query_mit_referenz_im_radius": {
        "radius_m": RADIUS_M,
        "anteil": float(n_loesbar / n_query),
        "anteil_selber_fotograf_selber_tag": float(je_query["gleicher_tag"].mean()),
        "anteil_anderer_fotograf": float(je_query["anderer_fotograf"].mean()),
        "median_nachbarn": float(je_query["n"].median()),
        "median_tage_zum_naechsten": float(je_query["tage_min"].median()),
    },
}
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
print(f"\n-> {AUDIT_PATH}")